# Tuning: depth × learning_rate on the tick channel

In [ ]:
import json

import boto3
import pandas as pd
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

session = LocalSession()
BUCKET = "electricity-price-streaming"
s3 = boto3.client(
    "s3", endpoint_url="http://electricity-price-streaming-moto:5000"
)
rows = []
for depth, lr in [(4, 0.1), (6, 0.1), (8, 0.05)]:
    est = Estimator(
        image_uri="electricity-price-streaming:train",
        role="arn:aws:iam::111111111111:role/service-role/SageMakerRole",
        instance_count=1,
        instance_type="local",
        output_path=f"s3://{BUCKET}/output",
        sagemaker_session=session,
        hyperparameters={
            "iterations": 250,
            "depth": depth,
            "learning_rate": lr,
        },
    )
    est.set_entry_point(
        source_dir="src/electricity_price_streaming", entry_point="train.py"
    )
    est.fit({"train": f"s3://{BUCKET}/input/train"})
    m = json.loads(
        s3.get_object(
            Bucket=BUCKET,
            Key=f"output/{est.latest_training_job.name}/output/metrics.json",
        )["Body"].read()
    )
    rows.append({"depth": depth, "lr": lr, **m})
pd.DataFrame(rows).sort_values("roc_auc", ascending=False)